In [ ]:
#!/usr/bin/env python3
"""
Compute Best‑Level Order‑Flow Imbalance (OFI) from raw limit‑order‑book updates.

Implements the three‑case rules of Cont et al. (2014) / Taranto & Zoudeh (2022)
for the best bid/ask (m = 1) and aggregates the resulting OFI into fixed‑width
time buckets (default: 1 minute).

---------------------------------------------------------------------------
USAGE
---------------------------------------------------------------------------
    python build_best_level_ofi.py               # uses defaults
    python build_best_level_ofi.py --in data.csv --bucket 30s
---------------------------------------------------------------------------

Author: “OFI Builder”
"""

from __future__ import annotations

import argparse
from pathlib import Path

import pandas as pd


# ---------------------------------------------------------------------------
# Configuration defaults (overridable from CLI)
# ---------------------------------------------------------------------------

IN_CSV_DEFAULT: Path = Path("lob.csv")
OUT_CSV_DEFAULT: Path = Path("best_level_ofi.csv")
BUCKET_DEFAULT: str = "1min"  # pandas offset alias, e.g. "30s", "5min", "1H"

# Column names expected in the input CSV
COL_TIME = "timestamp"
COL_BPX  = "bid_price"
COL_BSZ  = "bid_size"
COL_APX  = "ask_price"
COL_ASZ  = "ask_size"


# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def _parse_timestamp(series: pd.Series) -> pd.Series:
    """
    Convert the 'timestamp' column to pandas datetime64[ns, UTC].

    Handles either ISO‑8601 strings or integer Unix milliseconds.
    """
    # Numeric => treat as epoch‑ms
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_datetime(series.astype("int64"), unit="ms", utc=True)

    # Otherwise let pandas parse strings
    return pd.to_datetime(series, utc=True, errors="coerce")


def load_lob(path: Path) -> pd.DataFrame:
    """Read LOB CSV, parse timestamps, enforce dtypes, and sort chronologically."""
    df = pd.read_csv(
        path,
        usecols=[COL_TIME, COL_BPX, COL_BSZ, COL_APX, COL_ASZ],
        dtype={
            COL_BPX: "float64",
            COL_BSZ: "int64",
            COL_APX: "float64",
            COL_ASZ: "int64",
        },
    )

    # Timestamp handling
    df[COL_TIME] = _parse_timestamp(df[COL_TIME])

    # Drop rows with unparseable timestamps
    df = df.dropna(subset=[COL_TIME])

    # Sort to guarantee chronological order (handles out‑of‑order inputs)
    df = df.sort_values(COL_TIME, ignore_index=True)
    return df


def compute_best_level_ofi(df: pd.DataFrame, bucket: str = BUCKET_DEFAULT) -> pd.Series:
    """
    Vectorised computation of m=1 OFI and aggregation into fixed buckets.

    Parameters
    ----------
    df      : DataFrame with columns timestamp, bid/ask price & size.
    bucket  : pandas offset alias (e.g. "1min") for resampling frequency.

    Returns
    -------
    Series indexed by bucket end‑times, named 'best_level_ofi'.
    """
    prev = df.shift()  # previous book state for each update

    # ----- Bid flow (Cont et al. three‑case rule) --------------------------
    bid_flow = (
        (df[COL_BPX] > prev[COL_BPX]) * df[COL_BSZ] +
        (df[COL_BPX] == prev[COL_BPX]) * (df[COL_BSZ] - prev[COL_BSZ]) +
        (df[COL_BPX] < prev[COL_BPX]) * (-prev[COL_BSZ])
    )

    # ----- Ask flow --------------------------------------------------------
    ask_flow = (
        (df[COL_APX] > prev[COL_APX]) * (-df[COL_ASZ]) +
        (df[COL_APX] == prev[COL_APX]) * (df[COL_ASZ] - prev[COL_ASZ]) +
        (df[COL_APX] < prev[COL_APX]) * (prev[COL_ASZ])
    )

    # ----- Net event‑level OFI --------------------------------------------
    ofi_event = bid_flow - ask_flow

    # ----- Aggregate into fixed‑width buckets -----------------------------
    ofi_series = (
        ofi_event
        .rename("ofi_event")
        .to_frame()
        .set_index(df[COL_TIME])
        .resample(bucket, label="right", closed="right")
        .sum(min_count=1)                         # NaN if a bucket has no events
        .rename(columns={"ofi_event": "best_level_ofi"})
        .squeeze()
    )

    return ofi_series


# ---------------------------------------------------------------------------
# CLI / main
# ---------------------------------------------------------------------------

def _build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="Build Best‑Level OFI time series.")
    p.add_argument("--in",  dest="in_csv",  type=Path, default=IN_CSV_DEFAULT,
                   help=f"input LOB CSV (default: {IN_CSV_DEFAULT})")
    p.add_argument("--out", dest="out_csv", type=Path, default=OUT_CSV_DEFAULT,
                   help=f"output CSV for OFI series (default: {OUT_CSV_DEFAULT})")
    p.add_argument("--bucket", dest="bucket", default=BUCKET_DEFAULT,
                   help=f"resampling frequency, pandas offset alias (default: {BUCKET_DEFAULT})")
    return p


def main() -> None:
    args = _build_parser().parse_args()

    # 1. Load & clean data
    df = load_lob(args.in_csv)

    # 2. Compute OFI series
    ofi = compute_best_level_ofi(df, bucket=args.bucket)

    # 3. Persist & preview
    ofi.to_csv(args.out_csv, header=True)
    print(f"✅ Best‑Level OFI written to {args.out_csv} ({len(ofi)} rows)")
    print(ofi.head(10).to_string())


if __name__ == "__main__":
    main()
